# RAG Workbook: LangChain + ChromaDB

Retrieval-Augmented Generation (RAG) grounds an LLM's answer in documents you provide, reducing hallucinations and keeping responses factual. This notebook walks through the full pipeline: split documents into chunks, embed them into a vector store, retrieve the most relevant chunks for a query, and pass them to an LLM to generate a grounded answer. Everything runs in-memory — no external files, no database setup, works on Google Colab or locally.

In [ ]:
# Install dependencies only when running on Google Colab.
# Local users: activate your .venv — all packages are already in requirements.txt.
if "google.colab" in str(get_ipython()):
    import subprocess
    subprocess.run(["pip", "install", "-q",
                    "langchain", "langchain-openai", "langchain-chroma",
                    "langchain-text-splitters", "chromadb", "python-dotenv"])

In [ ]:
import os

# API key: Colab reads from secrets; local reads from .env file.
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

print("Setup complete. OPENAI_API_KEY present:", bool(os.environ.get("OPENAI_API_KEY")))

## Step 1: Load and chunk your documents

RAG works best when documents are split into small, focused chunks — typically 200–500 characters — so the retriever can surface only the most relevant pieces rather than entire files. `RecursiveCharacterTextSplitter` splits on natural boundaries (paragraphs, sentences, words) before falling back to raw character splits.

In [ ]:
raw_docs = [
    Document(
        page_content="LangGraph is a library for building stateful, multi-actor applications with LLMs. It extends LangChain with graph-based control flow, letting you define nodes and edges to orchestrate complex agent pipelines.",
        metadata={"source": "langgraph-intro"},
    ),
    Document(
        page_content="ChromaDB is an open-source vector database designed for AI applications. It stores embeddings alongside metadata and supports fast similarity search using cosine distance by default.",
        metadata={"source": "chromadb-intro"},
    ),
    Document(
        page_content="OpenAI's text-embedding-3-small model produces 1536-dimensional vectors and is optimized for retrieval tasks. It outperforms ada-002 on most benchmarks at a fraction of the cost.",
        metadata={"source": "openai-embeddings"},
    ),
    Document(
        page_content="Retrieval-Augmented Generation (RAG) combines a retriever with a language model. The retriever fetches relevant chunks from a knowledge base; the LLM uses those chunks as context to produce a grounded answer.",
        metadata={"source": "rag-overview"},
    ),
    Document(
        page_content="The RecursiveCharacterTextSplitter splits text hierarchically: first on double newlines, then single newlines, then spaces, then individual characters. This preserves semantic boundaries better than fixed-size splits.",
        metadata={"source": "text-splitter-docs"},
    ),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(raw_docs)

print(f"Documents: {len(raw_docs)} → Chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"  [{i}] ({chunk.metadata['source']}) {chunk.page_content[:70]}...")

## Step 2: Embed and store in ChromaDB

Each chunk is converted to a vector using OpenAI's `text-embedding-3-small` model. Those vectors are stored in an in-memory ChromaDB collection — no disk writes, no configuration, works on any platform. `Chroma.from_documents()` handles embedding and storage in one call.

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(chunks, embeddings)

print(f"Stored {vectorstore._collection.count()} chunks in ChromaDB (in-memory)")

## Step 3: Retrieve and generate

The retriever finds the top-k most similar chunks for a query using cosine similarity. Those chunks are injected into a prompt alongside the question, and `gpt-4o-mini` generates a grounded answer. `create_stuff_documents_chain` formats the context; `create_retrieval_chain` wires retrieval and generation into a single callable.